In [ ]:
pip install sentence-transformers chromadb


In [ ]:
pip install chromadb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [10]:
import json

input_path = "/content/drive/MyDrive/Major Project/finetune_data (step 4 result).jsonl"
output_path = "/content/drive/MyDrive/Major Project/chunked data new/chunk_data.jsonl"

def basic_chunk_text(text, max_chars=1000):
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]

with open(input_path, "r", encoding="utf-8") as infile, open(output_path, "w", encoding="utf-8") as outfile:
    for line in infile:
        if not line.strip():
            continue
        try:
            entry = json.loads(line)
            full_text = f"Gene: {entry['Gene.symbol']}\nGene Title: {entry['Gene.title']}\nRegulation: {entry['Regulation']}\n\nTitle: {entry['Title']}\n\nAbstract: {entry['Abstract']}"
            chunks = basic_chunk_text(full_text)
            for chunk in chunks:
                json.dump({"text": chunk}, outfile)
                outfile.write("\n")
        except KeyError:
            continue  # Skip entries missing keys

print("✅ Chunked data with 'text' key saved correctly.")

✅ Chunked data with 'text' key saved correctly.


In [11]:
from sentence_transformers import SentenceTransformer
import json
import pickle

chunked_path = "/content/drive/MyDrive/Major Project/chunked data new/chunk_data.jsonl"
embedding_path = "/content/drive/MyDrive/Major Project/chunked data new/fixed_embedding_data.pkl"

# Load and verify chunks
chunks = []
with open(chunked_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            entry = json.loads(line)
            if "text" in entry and entry["text"].strip():
                chunks.append(entry["text"])

print(f"Loaded {len(chunks)} text chunks.")

# Embed using SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks, show_progress_bar=True)

# Save for later steps
with open(embedding_path, "wb") as f:
    pickle.dump({"texts": chunks, "embeddings": embeddings}, f)

print("✅ Embeddings regenerated and saved.")

Loaded 2189 text chunks.


Batches:   0%|          | 0/69 [00:00<?, ?it/s]

✅ Embeddings regenerated and saved.


In [14]:
import chromadb
import pickle
import uuid

# Load embeddings from your pickle file
embedding_path = "/content/drive/MyDrive/Major Project/chunked data new/fixed_embedding_data.pkl"

with open(embedding_path, "rb") as f:
    data = pickle.load(f)

texts = data["texts"]
embeddings = data["embeddings"]

# ✅ Updated way to start Chroma without deprecated config
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Major Project/chroma_db_1")

# Create or get collection
collection = chroma_client.get_or_create_collection(name="gene_chunks")

# Generate unique IDs
ids = [str(uuid.uuid4()) for _ in texts]

# Add data to collection
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids
)

print("✅ Embeddings stored successfully in ChromaDB!")



✅ Embeddings stored successfully in ChromaDB!
